In [38]:
import pandas as pd
import geopandas as gpd
import re


1. PARAMÈTRES ET FICHIERS D'ENTRÉE


In [39]:

# Remplacer par le chemin de ton tableur (Excel ou CSV)
FICHIER_PERMIS = "PC_2026filtre.xlsx"  # ou "permis.csv"

# Chemin vers ton fichier GeoJSON cadastre
FICHIER_CADASTRE = "cadastre_historique_juin_2026_idu.geojson"

# Nom du fichier de sortie
FICHIER_SORTIE = "permis_geometries5.geojson"

# Noms des colonnes
COL_ID_PERMIS = "dossier"      # Remplace par le nom exact du champ identifiant le permis
COL_PARCELLES = "parcelles"      # Le champ avec les parcelles (ex: "578 AB 25, 578 AB 492")
COL_EMPLACEMENT = "Emplacement"  # Le champ du cadastre GeoJSON avec le code parcelle


2. CHARGEMENT ET NETTOYAGE


In [40]:

print("Chargement des données...")

# Chargement du tableur (utilise pd.read_csv si ton tableur est un CSV)
if FICHIER_PERMIS.endswith('.csv'):
    df_permis = pd.read_csv(FICHIER_PERMIS)
else:
    df_permis = pd.read_excel(FICHIER_PERMIS)

# Chargement du GeoJSON
cadastre = gpd.read_file(FICHIER_CADASTRE)

print("Données chargées !!!")


# Fonction de nettoyage : supprime espaces multiples et met en majuscules
def nettoyer_code(code_str):
    if pd.isna(code_str):
        return ""
    return re.sub(r'\s+', ' ', str(code_str)).strip().upper()

# Clean de la colonne du cadastre
cadastre['code_cadastre_clean'] = cadastre[COL_EMPLACEMENT].apply(nettoyer_code)

print("3️⃣ Extraction de TOUTES les parcelles par permis...")

lignes_association = []

for idx, row in df_permis.iterrows():
    permis_id = row[COL_ID_PERMIS]
    raw_parcelles = str(row[COL_PARCELLES]) if pd.notna(row[COL_PARCELLES]) else ""
    
    if not raw_parcelles.strip():
        continue
    
    # Découpage strict sur la virgule
    liste_codes = [c.strip() for c in raw_parcelles.split(',') if c.strip()]
    
    for code_brut in liste_codes:
        code_propre = nettoyer_code(code_brut)
        
        # Copie de la ligne du permis avec le code de parcelle nettoyé
        dict_ligne = row.to_dict()
        dict_ligne['code_parcelle_match'] = code_propre
        lignes_association.append(dict_ligne)

# Création de la table de correspondance 1 ligne = 1 parcelle d'1 permis
df_association = pd.DataFrame(lignes_association)

Chargement des données...
Données chargées !!!
3️⃣ Extraction de TOUTES les parcelles par permis...



3. JOINTURE ET FUSION (DISSOLVE)


In [41]:
print("4️⃣ Récupération des géométries dans le cadastre...")

# Jointure pour récupérer la géométrie de CHAQUE parcelle du permis
gdf_merged = cadastre.merge(
    df_association,
    left_on='code_cadastre_clean',
    right_on='code_parcelle_match',
    how="inner"
)

# Contrôle dans la console
parcelles_demandees = set(df_association['code_parcelle_match'])
parcelles_trouvees = set(gdf_merged['code_parcelle_match'])
manquantes = parcelles_demandees - parcelles_trouvees

print(f"\n📊 Bilan de correspondance :")
print(f"   - Nombre total d'associations permis <-> parcelle créées : {len(df_association)}")
print(f"   - Nombre de géométries de parcelles trouvées : {len(gdf_merged)}")

if manquantes:
    print(f"   ⚠️ {len(manquantes)} parcelle(s) n'ont pas été trouvées dans le cadastre GeoJSON !")
    print("   Exemples introuvables :", list(manquantes)[:10])

print("\n5️⃣ Fusion des géométries par permis (Dissolve)...")

# Agrégation (Union) de toutes les parcelles liées au même permis ID
permis_geometries = gdf_merged.dissolve(by=COL_ID_PERMIS, as_index=False)

# Nettoyage des colonnes techniques temporaires
cols_temp = ['code_cadastre_clean', 'code_parcelle_match']
permis_geometries = permis_geometries.drop(columns=[c for c in cols_temp if c in permis_geometries.columns])

4️⃣ Récupération des géométries dans le cadastre...

📊 Bilan de correspondance :
   - Nombre total d'associations permis <-> parcelle créées : 201
   - Nombre de géométries de parcelles trouvées : 196
   ⚠️ 5 parcelle(s) n'ont pas été trouvées dans le cadastre GeoJSON !
   Exemples introuvables : ['169 ZC 1', '227 AH 577', '259 AI 290', '254 AO 441', '381 I 696']

5️⃣ Fusion des géométries par permis (Dissolve)...


EXTRACTION DES DOSSIERS AVEC PARCELLES MANQUANTES

In [43]:
# Identification des parcelles non retrouvées dans le GeoJSON
df_manquants = df_association[~df_association['code_parcelle_match'].isin(gdf_merged['code_parcelle_match'])]

if not df_manquants.empty:
    # Agrégation par dossier
    dossiers_incomplets = (
        df_manquants.groupby(COL_ID_PERMIS)['code_parcelle_match']
        .apply(list)
        .reset_index()
    )
    dossiers_incomplets.columns = [COL_ID_PERMIS, 'parcelles_non_trouvees']
    
    print(f"❌ {len(dossiers_incomplets)} dossier(s) contiennent des parcelles non retrouvées :\n")
    
    for idx, row in dossiers_incomplets.iterrows():
        parcelles_str = ", ".join(row['parcelles_non_trouvees'])
        print(f"  • Dossier '{row[COL_ID_PERMIS]}' ➔ Parcelle(s) manquante(s) : {parcelles_str}")
        
    # Optionnel : Export de la liste en fichier Excel pour correction
    dossiers_incomplets.to_excel("dossiers_parcelles_manquantes.xlsx", index=False)
    print("\n📁 Liste exportée dans 'dossiers_parcelles_manquantes.xlsx'")
else:
    print("✅ Toutes les parcelles de tous les dossiers ont été trouvées !")

❌ 5 dossier(s) contiennent des parcelles non retrouvées :

  • Dossier 'PC 031 169 26 00009' ➔ Parcelle(s) manquante(s) : 169 ZC 1
  • Dossier 'PC 031 227 25 00012' ➔ Parcelle(s) manquante(s) : 227 AH 577
  • Dossier 'PC 031 254 26 00004' ➔ Parcelle(s) manquante(s) : 254 AO 441
  • Dossier 'PC 031 259 25 00008' ➔ Parcelle(s) manquante(s) : 259 AI 290
  • Dossier 'PC 031 381 26 00003' ➔ Parcelle(s) manquante(s) : 381 I 696

📁 Liste exportée dans 'dossiers_parcelles_manquantes.xlsx'


4. Export

In [42]:
print(f"6️⃣ Export vers {FICHIER_SORTIE}...")
permis_geometries.to_file(FICHIER_SORTIE, driver="GeoJSON")

print("\n✨ Traitement terminé avec succès !")

6️⃣ Export vers permis_geometries5.geojson...

✨ Traitement terminé avec succès !
